# 01 - Auditoría de datos: Online Retail II

Objetivo: entender la calidad de los datos, validar reglas de limpieza y
detectar patrones (cancelaciones, clientes mayoristas, ruido de sistema)
antes de definir el pipeline de etiquetado (`labeling`) y features.

Usa `src/data/loader.py` como única fuente de carga (con caché Parquet).


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.data.loader import load_online_retail

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


## 1. Carga de datos

In [2]:
df = load_online_retail()
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
df.head()


Filas: 1,067,371 | Columnas: 9


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1067371 non-null  str           
 1   StockCode     1067371 non-null  str           
 2   Description   1062989 non-null  str           
 3   Quantity      1067371 non-null  int64         
 4   InvoiceDate   1067371 non-null  datetime64[us]
 5   Price         1067371 non-null  float64       
 6   Customer ID   824364 non-null   float64       
 7   Country       1067371 non-null  str           
 8   source_sheet  1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 139.7 MB


## 2. Valores nulos

In [4]:
df.isna().sum()


Invoice              0
StockCode            0
Description       4382
Quantity             0
InvoiceDate          0
Price                0
Customer ID     243007
Country              0
source_sheet         0
dtype: int64

**Hallazgo esperado:** `Customer ID` concentra la mayoría de nulos (~243k).
Estas filas no se pueden atribuir a ningún cliente y no sirven para el
análisis de churn (que es a nivel cliente).

## 3. Cancelaciones (Invoice con prefijo 'C')

In [5]:
df["is_cancellation"] = df["Invoice"].str.startswith("C")

n_cancel = df["is_cancellation"].sum()
print(f"Filas de cancelación: {n_cancel:,} ({n_cancel/len(df)*100:.2f}%)")

neg_qty_no_c = ((df["Quantity"] < 0) & (~df["is_cancellation"])).sum()
pos_qty_c = ((df["Quantity"] > 0) & (df["is_cancellation"])).sum()

print(f"Quantity negativa SIN prefijo 'C': {neg_qty_no_c:,}")
print(f"Quantity positiva CON prefijo 'C': {pos_qty_c:,}")


Filas de cancelación: 19,494 (1.83%)
Quantity negativa SIN prefijo 'C': 3,457
Quantity positiva CON prefijo 'C': 1


In [6]:
# Filas con Quantity negativa y SIN prefijo C: ¿qué son?
noise = df[(df["Quantity"] < 0) & (~df["is_cancellation"])]
print(f"Total: {len(noise):,}")
print(f"Customer ID nulo en este subconjunto: {noise['Customer ID'].isna().sum()} de {len(noise)}")
noise["Description"].value_counts().head(15)


Total: 3,457
Customer ID nulo en este subconjunto: 3457 de 3457


Description
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
dotcom                      8
damages?                    7
??                          7
crushed                     6
given away                  6
Name: count, dtype: int64

**Hallazgo:** las quantities negativas sin prefijo `C` corresponden a
ajustes internos (`check`, `damages?`, `given away`, etc.), no a ventas
reales, y el 100% tiene `Customer ID` nulo. Se eliminan automáticamente al
filtrar por `Customer ID` no nulo.

## 4. Efecto de filtrar por Customer ID no nulo

In [7]:
df_with_customer = df[df["Customer ID"].notna()].copy()

print(f"Filas con Customer ID válido: {len(df_with_customer):,}")
print(f"Cancelaciones (prefijo C) en este subconjunto: {df_with_customer['is_cancellation'].sum():,}")
print(f"Quantity negativa SIN prefijo C en este subconjunto: "
      f"{((df_with_customer['Quantity'] < 0) & (~df_with_customer['is_cancellation'])).sum():,}")


Filas con Customer ID válido: 824,364
Cancelaciones (prefijo C) en este subconjunto: 18,744
Quantity negativa SIN prefijo C en este subconjunto: 0


**Conclusión:** filtrar por `Customer ID` no nulo elimina todo el ruido de
quantity negativa sin cancelación. Las cancelaciones legítimas (prefijo `C`)
se mantienen y se filtran aparte según la necesidad del análisis.

## 5. Transacciones válidas: distribución de Quantity y Price

In [8]:
df_valid = df_with_customer[~df_with_customer["is_cancellation"]].copy()

print("Quantity - describe:")
print(df_valid["Quantity"].describe(percentiles=[.5, .9, .95, .99, .999]))

print("\nPrice - describe:")
print(df_valid["Price"].describe(percentiles=[.5, .9, .95, .99, .999]))

print(f"\nFilas con Price <= 0: {(df_valid['Price'] <= 0).sum():,}")
print(f"Filas con Quantity <= 0: {(df_valid['Quantity'] <= 0).sum():,}")


Quantity - describe:
count    805620.000000
mean         13.307665
std         144.306739
min           1.000000
50%           5.000000
90%          24.000000
95%          36.000000
99%         128.000000
99.9%       576.000000
max       80995.000000
Name: Quantity, dtype: float64

Price - describe:
count    805620.000000
mean          3.206279
std          29.197901
min           0.000000
50%           1.950000
90%           6.750000
95%           8.500000
99%          14.950000
99.9%        49.950000
max       10953.500000
Name: Price, dtype: float64

Filas con Price <= 0: 71
Filas con Quantity <= 0: 0


In [9]:
# Ruido de sistema: TEST001
test_rows = df_valid[df_valid["StockCode"] == "TEST001"]
print(f"Filas TEST001: {len(test_rows)}")
test_rows[["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]]


Filas TEST001: 11


,Invoice,StockCode,Description,Quantity,Price,Customer ID
27994,491725,TEST001,This is a test product.,10,4.5,12346.0
28251,491742,TEST001,This is a test product.,5,4.5,12346.0
28254,491744,TEST001,This is a test product.,5,4.5,12346.0
39398,492718,TEST001,This is a test product.,5,4.5,12346.0
45228,493410,TEST001,This is a test product.,5,4.5,12346.0
45230,493412,TEST001,This is a test product.,5,4.5,12346.0
56117,494450,TEST001,This is a test product.,5,4.5,12346.0
66084,495295,TEST001,This is a test product.,5,4.5,12346.0
89084,497819,TEST001,This is a test product.,5,0.0,14103.0
89180,497843,TEST001,This is a test product.,5,0.0,14827.0


**Hallazgo:** `TEST001` son productos de prueba del sistema, no ventas
reales. Se excluyen del dataset limpio.

## 6. Perfil de clientes: identificación de mayoristas (p99)

In [10]:
customer_profile = (
    df_valid[df_valid["StockCode"] != "TEST001"]
    .assign(spend=lambda d: d["Quantity"] * d["Price"])
    .groupby("Customer ID")
    .agg(
        total_quantity=("Quantity", "sum"),
        total_spend=("spend", "sum"),
        n_invoices=("Invoice", "nunique"),
        n_countries=("Country", "nunique"),
    )
    .reset_index()
)

print(customer_profile[["total_quantity", "total_spend", "n_invoices"]]
      .describe(percentiles=[.5, .9, .95, .99, .999]))


       total_quantity    total_spend   n_invoices
count     5879.000000    5879.000000  5879.000000
mean      1823.585814    3018.065007     6.287464
std       8970.038299   14736.356702    13.014585
min          1.000000       0.000000     1.000000
50%        492.000000     898.870000     3.000000
90%       3236.200000    5595.230000    13.000000
95%       5446.700000    9533.584000    21.000000
99%      17505.760000   29721.812800    46.000000
99.9%   123386.882000  178150.819720   160.392000
max     367833.000000  608821.650000   398.000000


In [11]:
q_threshold = customer_profile["total_quantity"].quantile(0.99)
spend_threshold = customer_profile["total_spend"].quantile(0.99)

customer_profile["is_wholesale"] = (
    (customer_profile["total_quantity"] > q_threshold)
    | (customer_profile["total_spend"] > spend_threshold)
)

n_wholesale = customer_profile["is_wholesale"].sum()
print(f"Umbral Quantity (p99): {q_threshold:,.0f}")
print(f"Umbral Spend (p99): {spend_threshold:,.2f}")
print(f"Clientes marcados como mayoristas: {n_wholesale} ({n_wholesale/len(customer_profile)*100:.2f}%)")


Umbral Quantity (p99): 17,506
Umbral Spend (p99): 29,721.81
Clientes marcados como mayoristas: 70 (1.19%)


In [12]:
customer_profile[customer_profile["is_wholesale"]].sort_values("total_spend", ascending=False).head(20)


,Customer ID,total_quantity,total_spend,n_invoices,n_countries,is_wholesale
5693,18102.0,188340,608821.65,145,1,True
2278,14646.0,367833,528602.52,152,1,True
1790,14156.0,165992,313946.37,156,1,True
2539,14911.0,150237,295972.63,398,1,True
5051,17450.0,85368,246973.09,51,1,True
1332,13694.0,189205,196482.81,143,1,True
5110,17511.0,119656,175603.55,60,1,True
4062,16446.0,80997,168472.50,2,1,True
4296,16684.0,104810,147142.77,55,1,True
68,12415.0,91743,144458.37,28,1,True


**Decisión pendiente para el pipeline de labeling/features:** definir si
los clientes mayoristas se excluyen del modelo de churn, se modelan aparte,
o se incluyen con una feature `is_wholesale`. Se retoma en
`src/labeling/`.

## 7. Resumen de reglas de limpieza validadas

1. Filtrar `Customer ID` no nulo → elimina ruido de quantity negativa sin
   cancelación (ajustes internos, no ventas).
2. Excluir `StockCode == 'TEST001'` → ruido de sistema.
3. Tratar `Invoice` con prefijo `'C'` como cancelación (no como venta).
4. Marcar clientes con `total_quantity` o `total_spend` por encima del
   percentil 99 como `is_wholesale` (umbral de referencia:
   Quantity > 17,506 · Spend > 29,704.60).

Estas reglas se implementarán como funciones reutilizables en
`src/data/` (limpieza) y `src/labeling/` (definición de churn), no
repetidas manualmente en cada notebook.